### **Restore the exact evaluator and model**

In [1]:
from pathlib import Path
import hashlib, json, re
import numpy as np
import pandas as pd
import sacrebleu
import torch

from IPython.display import display
from tqdm.auto import tqdm

NB = Path(
    "/home/mabdallah/alexandriax_mt_14d/notebooks/"
    "Inference_Variants_After_Continuation.ipynb"
)

if not NB.is_file():
    raise FileNotFoundError(NB)

nb = json.loads(NB.read_text(encoding="utf-8"))


def run_original(marker):
    hits = [
        "".join(cell.get("source", []))
        for cell in nb["cells"]
        if cell.get("cell_type") == "code"
        and marker in "".join(cell.get("source", []))
    ]

    if len(hits) != 1:
        raise RuntimeError(
            f"{marker!r}: found {len(hits)} cells"
        )

    exec(
        compile(hits[0], str(NB), "exec"),
        globals(),
    )


for marker in (
    "PROJECT_DIR = Path",
    "def read_table(path):",
    "def clean_string(value):",
    "def format_previous_context(row, include_speakers):",
    "dtype = torch.bfloat16 if torch.cuda.is_available()",
):
    run_original(marker)

tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

model.eval()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id

print("Locked40 rows:", len(locked40_df))
print("Model type:", getattr(model.config, "model_type", "?"))
print("Chat template available:", bool(tokenizer.chat_template))
print("Loaded adapters:", list(model.peft_config))

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 5.870262622833252
Locked40 rows: 5772
Model type: qwen2
Chat template available: True
Loaded adapters: ['cont_5200']


### **Routes, incumbent predictions, and fixed subset**

In [2]:
PRIVATE100 = (
    PROJECT_DIR
    / "inference_variants/"
      "100_official_private_test_"
      "continuation_router_threshold015_beam4_v1/"
      "turn_predictions.csv"
)

if not PRIVATE100.is_file():
    raise FileNotFoundError(PRIVATE100)

NAME = "system103_r2_locked40_fixed_subset_v1"
OUT = STAGE_ROOT / NAME
OUT.mkdir(parents=True, exist_ok=True)

# adapter key, continuation step, prompt variant
S100 = {
    "EG": ("cont_5200", 5200, "V05"),
    "JO": ("cont_5200", 5200, "V01"),
    "LB": ("cont_5200", 5200, "V01"),
    "LY": ("cont_6400", 6400, "V05"),
    "MA": ("cont_5200", 5200, "V03"),
    "MR": ("cont_2000", 2000, "V01"),
    "OM": ("cont_4900", 4900, "V01"),
    "PS": ("cont_5200", 5200, "V01"),
    "SA": ("cont_1600", 1600, "V03"),
    "SD": ("cont_5200", 5200, "V01"),
    "SY": ("cont_5200", 5200, "V05"),
    "TN": ("cont_7600", 7600, "V01"),
    "YE": ("cont_4900", 4900, "V01"),
}

EXT = {
    "LY": {
        "key": "ext_ly_400",
        "step": 400,
        "variant": "V05",
        "adapter": (
            PROJECT_DIR
            / "runs/country_external_specialists/"
              "LY_smol_aux_from_cont6400_lr5e6_2epochs_v1/"
              "checkpoint-400"
        ),
        "pred": (
            STAGE_ROOT
            / "extspec_ly_step000400_v05_"
              "exact_submission_route_v5/"
              "turn_predictions.csv"
        ),
    },
    "SD": {
        "key": "ext_sd_700",
        "step": 700,
        "variant": "V01",
        "adapter": (
            PROJECT_DIR
            / "runs/country_external_specialists/"
              "SD_smol_aux_from_cont5200_lr5e6_2epochs_v1/"
              "checkpoint-700"
        ),
        "pred": (
            STAGE_ROOT
            / "extspec_sd_step000700_v01_"
              "exact_submission_route_v5/"
              "turn_predictions.csv"
        ),
    },
}

# R2 uses the current best route.
ROUTE = S100.copy()

for country, spec in EXT.items():
    ADAPTER_PATHS[spec["key"]] = spec["adapter"]
    ROUTE[country] = (
        spec["key"],
        spec["step"],
        spec["variant"],
    )

for country, (adapter_key, _, _) in ROUTE.items():
    adapter_path = Path(ADAPTER_PATHS[adapter_key])

    if not (
        adapter_path / "adapter_config.json"
    ).is_file():
        raise FileNotFoundError(
            f"{country}: {adapter_path}"
        )


def load_predictions(path, required_ids):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(path)

    frame = pd.read_csv(
        path,
        keep_default_na=False,
    )[["source_id", "prediction"]]

    frame["source_id"] = (
        frame["source_id"].astype(str)
    )
    frame["prediction"] = (
        frame["prediction"]
        .astype(str)
        .str.strip()
    )

    frame = frame[
        frame["source_id"].isin(required_ids)
    ]

    if (
        frame["source_id"].duplicated().any()
        or set(frame["source_id"]) != required_ids
    ):
        raise RuntimeError(
            f"ID mismatch: {path}"
        )

    if frame["prediction"].eq("").any():
        raise RuntimeError(
            f"Empty prediction: {path}"
        )

    return frame.set_index(
        "source_id"
    )["prediction"]


locked = (
    locked40_df
    .sort_values("_row_order")
    .copy()
)

locked["source_id"] = (
    locked["source_id"].astype(str)
)
locked["country"] = (
    locked["country"]
    .astype(str)
    .str.upper()
)

baseline = locked[
    [
        "source_id",
        "country",
        "conversation_id",
        "reference_arabic",
        "_row_order",
    ]
].copy()

baseline["system100_prediction"] = ""

for country, (_, step, variant) in S100.items():
    mask = baseline["country"].eq(country)
    required_ids = set(
        baseline.loc[mask, "source_id"]
    )

    path = (
        STAGE_ROOT
        / f"c{step}_{variant.lower()}"
        / "turn_predictions.csv"
    )

    predictions = load_predictions(
        path,
        required_ids,
    )

    baseline.loc[
        mask,
        "system100_prediction",
    ] = (
        baseline.loc[mask, "source_id"]
        .map(predictions)
        .values
    )

baseline["incumbent_prediction"] = (
    baseline["system100_prediction"]
)

# Apply frozen LY/SD specialists.
for country, spec in EXT.items():
    mask = baseline["country"].eq(country)
    required_ids = set(
        baseline.loc[mask, "source_id"]
    )

    predictions = load_predictions(
        spec["pred"],
        required_ids,
    )

    baseline.loc[
        mask,
        "incumbent_prediction",
    ] = (
        baseline.loc[mask, "source_id"]
        .map(predictions)
        .values
    )


def macro_score(frame, prediction_column):
    bleu_scores = []
    chrf_scores = []

    for _, part in frame.groupby(
        "country",
        sort=True,
    ):
        hypotheses = (
            part[prediction_column]
            .astype(str)
            .tolist()
        )
        references = (
            part["reference_arabic"]
            .astype(str)
            .tolist()
        )

        bleu_scores.append(
            sacrebleu.corpus_bleu(
                hypotheses,
                [references],
                tokenize="flores200",
            ).score
        )

        chrf_scores.append(
            sacrebleu.corpus_chrf(
                hypotheses,
                [references],
                word_order=2,
            ).score
        )

    return (
        float(np.mean(bleu_scores)),
        float(np.mean(chrf_scores)),
    )


system100_full = macro_score(
    baseline,
    "system100_prediction",
)

incumbent_full = macro_score(
    baseline,
    "incumbent_prediction",
)

np.testing.assert_allclose(
    system100_full,
    (30.182420, 44.805134),
    atol=5e-5,
)

np.testing.assert_allclose(
    incumbent_full[0],
    30.317100,
    atol=5e-5,
)

# Deterministic complete-conversation subset:
# approximately 80 turns per country.
TARGET_TURNS = 80
SUBSET_FILE = OUT / "fixed_subset_rows.csv"

if SUBSET_FILE.is_file():
    subset_ids = set(
        pd.read_csv(
            SUBSET_FILE,
            dtype={"source_id": str},
        )["source_id"].astype(str)
    )

else:
    subset_ids = set()

    for country in COUNTRIES:
        part = locked[
            locked["country"].eq(country)
        ]

        conversations = (
            part.groupby("conversation_id")
            .size()
            .rename("turns")
            .reset_index()
        )

        conversations["_hash"] = (
            conversations["conversation_id"]
            .map(
                lambda conversation_id:
                hashlib.sha256(
                    (
                        f"R2-v1|{country}|"
                        f"{conversation_id}"
                    ).encode()
                ).hexdigest()
            )
        )

        selected = []
        selected_turns = 0

        for row in (
            conversations
            .sort_values("_hash")
            .itertuples(index=False)
        ):
            selected.append(
                str(row.conversation_id)
            )
            selected_turns += int(row.turns)

            if selected_turns >= TARGET_TURNS:
                break

        subset_ids |= set(
            part.loc[
                part["conversation_id"]
                .astype(str)
                .isin(selected),
                "source_id",
            ].astype(str)
        )

    locked.loc[
        locked["source_id"].isin(subset_ids),
        [
            "source_id",
            "country",
            "conversation_id",
            "_row_order",
        ],
    ].sort_values(
        "_row_order"
    ).to_csv(
        SUBSET_FILE,
        index=False,
    )

subset = (
    locked[
        locked["source_id"].isin(subset_ids)
    ]
    .merge(
        baseline[
            [
                "source_id",
                "system100_prediction",
                "incumbent_prediction",
            ]
        ],
        on="source_id",
        validate="one_to_one",
    )
    .sort_values("_row_order")
    .reset_index(drop=True)
)

assert set(subset["country"]) == set(COUNTRIES)
assert subset["source_id"].is_unique

print(
    "Private System100 identified but "
    "not mixed with locked40:"
)
print(PRIVATE100)

print(
    "Full locked40 System100:",
    tuple(round(x, 6) for x in system100_full),
)
print(
    "Full locked40 current incumbent:",
    tuple(round(x, 6) for x in incumbent_full),
)

display(
    subset.groupby(
        "country",
        as_index=False,
    ).agg(
        turns=("source_id", "size"),
        conversations=(
            "conversation_id",
            "nunique",
        ),
    )
)

Private System100 identified but not mixed with locked40:
/home/mabdallah/alexandriax_mt_14d/inference_variants/100_official_private_test_continuation_router_threshold015_beam4_v1/turn_predictions.csv
Full locked40 System100: (30.18242, 44.805134)
Full locked40 current incumbent: (30.3171, 44.898931)


,country,turns,conversations
0,EG,81,26
1,JO,80,27
2,LB,80,25
3,LY,80,29
4,MA,81,25
5,MR,83,26
6,OM,81,27
7,PS,81,28
8,SA,82,27
9,SD,83,25


### **R2 adapter-on reasoning planner**

In [3]:
PLANNER_FILE = OUT / "planner_audit.csv"


def build_planner_prompt(row):
    user_content = f"""Audit this translation conservatively.

Country/dialect:
{clean_string(row["country"])} / {clean_string(row["dialect"])}

Domain:
{clean_string(row["domain"])}

Participants:
{clean_string(row["participants"])}

Current speaker:
{clean_string(row["speaker"])}

Speaker-to-addressee gender:
{clean_string(row["gender_direction"])}

Previous English context:
{format_previous_context(row, include_speakers=True)}

Current English:
{clean_string(row["source_text"])}

Existing Arabic translation:
{clean_string(row["incumbent_prediction"])}

Check only definite errors involving:
- Meaning or idiom.
- Omission or addition.
- Referent, speaker or gender.
- Entity or number preservation.
- Target-country dialect.

Valid paraphrases, spelling alternatives and optional
wording are not errors.

Reason briefly, then finish exactly with:

DECISION: KEEP or FIX
CONFIDENCE: HIGH or MEDIUM or LOW
ISSUES: one short line
PLAN: one short line

Use FIX with HIGH confidence only when a correction
is definitely necessary.

/think"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a conservative context-aware "
                "dialectal-Arabic translation auditor. "
                "Do not output a replacement translation."
            ),
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    if tokenizer.chat_template:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )

        except Exception:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

    return (
        "### System:\n"
        f"{messages[0]['content']}\n\n"
        "### Instruction:\n"
        f"{user_content}\n\n"
        "### Audit:\n"
    )


@torch.inference_mode()
def generate_planner(prompts):
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    device = next(model.parameters()).device

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    prompt_length = (
        encoded["input_ids"].shape[1]
    )

    output = model.generate(
        **encoded,
        max_new_tokens=192,
        do_sample=False,
        num_beams=1,
        repetition_penalty=1.05,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = output[:, prompt_length:]

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=False,
    )


def parse_planner(row, raw_output):
    raw_output = str(raw_output)

    for token in (
        "<|im_end|>",
        "<|endoftext|>",
        tokenizer.eos_token,
        tokenizer.pad_token,
    ):
        if token:
            raw_output = raw_output.replace(
                token,
                "",
            )

    # Incomplete native reasoning is never trusted.
    if (
        "<think>" in raw_output
        and "</think>" not in raw_output
    ):
        return {
            "decision": "KEEP",
            "confidence": "LOW",
            "apply_fix": False,
            "issues": "incomplete reasoning",
            "plan": "keep incumbent",
            "raw_planner": raw_output,
        }

    visible = (
        raw_output
        .rsplit("</think>", 1)[-1]
        .strip()
    )

    decisions = re.findall(
        r"DECISION\s*:\s*(KEEP|FIX)",
        visible,
        flags=re.I,
    )

    confidences = re.findall(
        r"CONFIDENCE\s*:\s*"
        r"(HIGH|MEDIUM|LOW)",
        visible,
        flags=re.I,
    )

    decision = (
        decisions[-1].upper()
        if decisions
        else "KEEP"
    )

    confidence = (
        confidences[-1].upper()
        if confidences
        else "LOW"
    )

    def extract_field(name, default):
        match = re.search(
            rf"{name}\s*:\s*(.*?)"
            rf"(?=\n(?:DECISION|CONFIDENCE|"
            rf"ISSUES|PLAN)\s*:|\Z)",
            visible,
            flags=re.I | re.S,
        )

        if not match:
            return default

        return " ".join(
            match.group(1).split()
        )

    return {
        "decision": decision,
        "confidence": confidence,
        "apply_fix": (
            decision == "FIX"
            and confidence == "HIGH"
        ),
        "issues": extract_field(
            "ISSUES",
            "none stated",
        ),
        "plan": extract_field(
            "PLAN",
            "keep incumbent",
        ),
        "raw_planner": raw_output,
    }


def run_routed_resumable(
    target,
    output_path,
    prompt_function,
    generation_function,
    parser_function,
    batch_size,
):
    required_ids = set(
        target["source_id"].astype(str)
    )

    if output_path.is_file():
        completed_frame = pd.read_csv(
            output_path,
            dtype={"source_id": str},
            keep_default_na=False,
        )

        completed_frame = (
            completed_frame[
                completed_frame["source_id"]
                .isin(required_ids)
            ]
            .drop_duplicates(
                "source_id",
                keep="last",
            )
        )

    else:
        completed_frame = pd.DataFrame()

    completed_ids = (
        set(
            completed_frame[
                "source_id"
            ].astype(str)
        )
        if len(completed_frame)
        else set()
    )

    pending_records = []

    for country in COUNTRIES:
        adapter_key = ROUTE[country][0]

        # Adapter is active during reasoning.
        activate_adapter(adapter_key)

        country_rows = target[
            target["country"].eq(country)
            & ~target["source_id"]
            .isin(completed_ids)
        ]

        for start in tqdm(
            range(
                0,
                len(country_rows),
                batch_size,
            ),
            desc=f"{output_path.stem} {country}",
        ):
            batch = country_rows.iloc[
                start:start + batch_size
            ]

            prompts = [
                prompt_function(row)
                for _, row in batch.iterrows()
            ]

            outputs = generation_function(
                prompts
            )

            for (_, row), output in zip(
                batch.iterrows(),
                outputs,
            ):
                pending_records.append({
                    "source_id": str(
                        row["source_id"]
                    ),
                    "country": country,
                    "adapter_key": adapter_key,
                    **parser_function(
                        row,
                        output,
                    ),
                })

            if len(pending_records) >= 20:
                completed_frame = pd.concat(
                    [
                        completed_frame,
                        pd.DataFrame(
                            pending_records
                        ),
                    ],
                    ignore_index=True,
                ).drop_duplicates(
                    "source_id",
                    keep="last",
                )

                atomic_csv(
                    completed_frame,
                    output_path,
                )

                pending_records = []

    if pending_records:
        completed_frame = pd.concat(
            [
                completed_frame,
                pd.DataFrame(
                    pending_records
                ),
            ],
            ignore_index=True,
        ).drop_duplicates(
            "source_id",
            keep="last",
        )

        atomic_csv(
            completed_frame,
            output_path,
        )

    if (
        set(
            completed_frame[
                "source_id"
            ].astype(str)
        )
        != required_ids
    ):
        raise RuntimeError(
            f"{output_path.name} incomplete. "
            "Rerun this cell to resume."
        )

    return completed_frame


planner = run_routed_resumable(
    target=subset,
    output_path=PLANNER_FILE,
    prompt_function=build_planner_prompt,
    generation_function=generate_planner,
    parser_function=parse_planner,
    batch_size=2,
)

planner["apply_fix"] = (
    planner["apply_fix"]
    .astype(str)
    .str.lower()
    .eq("true")
)

display(
    planner.groupby(
        [
            "decision",
            "confidence",
            "apply_fix",
        ],
        as_index=False,
    ).size()
)

display(
    planner.groupby(
        "country",
        as_index=False,
    ).agg(
        rows=("source_id", "size"),
        high_confidence_fixes=(
            "apply_fix",
            "sum",
        ),
    )
)

print(
    "High-confidence FIX rate:",
    f"{100 * planner['apply_fix'].mean():.2f}%",
)

planner_audit EG:   0%|          | 0/41 [00:00<?, ?it/s]

planner_audit JO:   0%|          | 0/40 [00:00<?, ?it/s]

planner_audit LB:   0%|          | 0/40 [00:00<?, ?it/s]

planner_audit LY:   0%|          | 0/40 [00:00<?, ?it/s]

planner_audit MA:   0%|          | 0/41 [00:00<?, ?it/s]

...

planner_audit TN:   0%|          | 0/41 [00:00<?, ?it/s]

planner_audit YE:   0%|          | 0/41 [00:00<?, ?it/s]

,decision,confidence,apply_fix,size
0,KEEP,LOW,False,1056


,country,rows,high_confidence_fixes
0,EG,81,0
1,JO,80,0
2,LB,80,0
3,LY,80,0
4,MA,81,0
5,MR,83,0
6,OM,81,0
7,PS,81,0
8,SA,82,0
9,SD,83,0


High-confidence FIX rate: 0.00%


### **Same-adapter Beam-4 correction**

In [4]:
CORRECTION_FILE = OUT / "corrections.csv"

audit_lookup = (
    planner[
        planner["apply_fix"]
    ]
    .set_index("source_id")
    .to_dict("index")
)

fix_rows = subset[
    subset["source_id"].isin(
        set(audit_lookup)
    )
].copy()


def build_correction_prompt(row):
    audit = audit_lookup[
        str(row["source_id"])
    ]

    route_variant = ROUTE[
        str(row["country"])
    ][2]

    direct_prompt = build_prompt(
        row,
        variant=route_variant,
        retrieval_mode="new",
    )

    direct_prompt = direct_prompt.rsplit(
        "### Arabic translation:",
        1,
    )[0].rstrip()

    return f"""{direct_prompt}

Existing Arabic draft:
{clean_string(row["incumbent_prediction"])}

High-confidence audit issue:
{clean_string(audit["issues"])}

Minimal correction plan:
{clean_string(audit["plan"])}

Additional rules:
- Change only the explicit error.
- Do not paraphrase unaffected wording.
- If the correction is not clearly justified, copy the draft exactly.
- Return only the final Arabic translation.

### Arabic translation:
"""


def parse_correction(row, output):
    return {
        "prediction": str(output).strip()
    }


if len(fix_rows):
    corrections = run_routed_resumable(
        target=fix_rows,
        output_path=CORRECTION_FILE,
        prompt_function=build_correction_prompt,
        generation_function=generate_batch,
        parser_function=parse_correction,
        batch_size=GEN_BATCH_SIZE,
    )

else:
    corrections = pd.DataFrame(
        columns=[
            "source_id",
            "country",
            "prediction",
        ]
    )

print(
    "Corrections completed:",
    len(corrections),
)

Corrections completed: 0


### **Score R2 against the existing systems**

In [5]:
ARABIC_PATTERN = re.compile(
    r"[\u0600-\u06ff"
    r"\u0750-\u077f"
    r"\u08a0-\u08ff]"
)

final = subset[
    [
        "source_id",
        "country",
        "conversation_id",
        "reference_arabic",
        "system100_prediction",
        "incumbent_prediction",
    ]
].copy()

final["r2_prediction"] = (
    final["incumbent_prediction"]
)
final["invalid_fallback"] = False

if len(corrections):
    correction_map = (
        corrections
        .set_index("source_id")[
            "prediction"
        ]
    )

    for index, row in final.iterrows():
        source_id = str(row["source_id"])

        if source_id not in correction_map:
            continue

        candidate = str(
            correction_map[source_id]
        ).strip()

        if (
            candidate
            and ARABIC_PATTERN.search(candidate)
        ):
            final.at[
                index,
                "r2_prediction",
            ] = candidate

        else:
            final.at[
                index,
                "invalid_fallback",
            ] = True

final["changed"] = (
    final["r2_prediction"]
    .astype(str)
    .str.strip()
    != final["incumbent_prediction"]
    .astype(str)
    .str.strip()
)


def score_system(
    frame,
    prediction_column,
    system_name,
):
    records = []

    for country, part in frame.groupby(
        "country",
        sort=True,
    ):
        hypotheses = (
            part[prediction_column]
            .astype(str)
            .tolist()
        )

        references = (
            part["reference_arabic"]
            .astype(str)
            .tolist()
        )

        records.append({
            "system": system_name,
            "country": country,
            "rows": len(part),
            "spBLEU": (
                sacrebleu.corpus_bleu(
                    hypotheses,
                    [references],
                    tokenize="flores200",
                ).score
            ),
            "chrF++": (
                sacrebleu.corpus_chrf(
                    hypotheses,
                    [references],
                    word_order=2,
                ).score
            ),
        })

    return pd.DataFrame(records)


system100_metrics = score_system(
    final,
    "system100_prediction",
    "System100_subset",
)

incumbent_metrics = score_system(
    final,
    "incumbent_prediction",
    "Current_incumbent_subset",
)

r2_metrics = score_system(
    final,
    "r2_prediction",
    "R2_subset",
)


def summarize(metrics):
    return {
        "system": metrics["system"].iloc[0],
        "rows": int(
            metrics["rows"].sum()
        ),
        "macro_spBLEU": float(
            metrics["spBLEU"].mean()
        ),
        "macro_chrF++": float(
            metrics["chrF++"].mean()
        ),
    }


comparison = pd.DataFrame([
    summarize(system100_metrics),
    summarize(incumbent_metrics),
    summarize(r2_metrics),
])

incumbent_bleu = float(
    incumbent_metrics["spBLEU"].mean()
)
incumbent_chrf = float(
    incumbent_metrics["chrF++"].mean()
)

comparison[
    "spBLEU_vs_incumbent"
] = (
    comparison["macro_spBLEU"]
    - incumbent_bleu
)

comparison[
    "chrF++_vs_incumbent"
] = (
    comparison["macro_chrF++"]
    - incumbent_chrf
)

country_comparison = (
    r2_metrics[
        [
            "country",
            "spBLEU",
            "chrF++",
        ]
    ]
    .merge(
        incumbent_metrics[
            [
                "country",
                "spBLEU",
                "chrF++",
            ]
        ],
        on="country",
        suffixes=(
            "_R2",
            "_incumbent",
        ),
        validate="one_to_one",
    )
)

country_comparison["spBLEU_delta"] = (
    country_comparison["spBLEU_R2"]
    - country_comparison[
        "spBLEU_incumbent"
    ]
)

country_comparison["chrF++_delta"] = (
    country_comparison["chrF++_R2"]
    - country_comparison[
        "chrF++_incumbent"
    ]
)

macro_gain = float(
    r2_metrics["spBLEU"].mean()
    - incumbent_bleu
)

harmed_countries = int(
    country_comparison[
        "spBLEU_delta"
    ].lt(-1e-9).sum()
)

worst_country_delta = float(
    country_comparison[
        "spBLEU_delta"
    ].min()
)

invalid_fallbacks = int(
    final["invalid_fallback"].sum()
)

max_invalid = max(
    1,
    int(0.01 * len(final)),
)

screen_passed = (
    macro_gain >= 0.20
    and harmed_countries <= 4
    and worst_country_delta >= -0.60
    and invalid_fallbacks <= max_invalid
)

display(
    comparison.round(6)
)

display(
    country_comparison
    .sort_values(
        "spBLEU_delta",
        ascending=False,
    )
    .reset_index(drop=True)
    .round(6)
)

print(
    "High-confidence fixes:",
    len(fix_rows),
)
print(
    "Actually changed translations:",
    int(final["changed"].sum()),
)
print(
    "Invalid correction fallbacks:",
    invalid_fallbacks,
)
print(
    f"Macro spBLEU gain: "
    f"{macro_gain:+.6f}"
)
print(
    "Harmed countries:",
    harmed_countries,
)
print(
    f"Worst country delta: "
    f"{worst_country_delta:+.6f}"
)
print()

print(
    "PASS — RUN R2 ON FULL LOCKED40"
    if screen_passed
    else
    "STOP — DO NOT RUN FULL LOCKED40 "
    "OR PRIVATE TEST"
)

atomic_csv(
    final,
    OUT / "r2_subset_predictions.csv",
)

atomic_csv(
    comparison,
    OUT / "system_comparison.csv",
)

atomic_csv(
    country_comparison,
    OUT / "country_comparison.csv",
)

manifest = {
    "system": NAME,
    "method": (
        "adapter-on reasoning planner "
        "plus same-adapter correction"
    ),
    "subset_rows": int(len(final)),
    "macro_spBLEU_gain": macro_gain,
    "harmed_countries": harmed_countries,
    "worst_country_delta": (
        worst_country_delta
    ),
    "high_confidence_fixes": int(
        len(fix_rows)
    ),
    "changed_translations": int(
        final["changed"].sum()
    ),
    "screen_passed": bool(
        screen_passed
    ),
    "private_system100_path": str(
        PRIVATE100
    ),
}

(
    OUT / "manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", OUT)

,system,rows,macro_spBLEU,macro_chrF++,spBLEU_vs_incumbent,chrF++_vs_incumbent
0,System100_subset,1056,29.635735,44.246884,-0.066313,-0.072094
1,Current_incumbent_subset,1056,29.702048,44.318978,0.000000,0.000000
2,R2_subset,1056,29.702048,44.318978,0.000000,0.000000


,country,spBLEU_R2,chrF++_R2,spBLEU_incumbent,chrF++_incumbent,spBLEU_delta,chrF++_delta
0,EG,30.344392,44.535626,30.344392,44.535626,0.0,0.0
1,JO,34.384817,48.333708,34.384817,48.333708,0.0,0.0
2,LB,32.707480,46.225587,32.707480,46.225587,0.0,0.0
3,LY,22.574917,38.404611,22.574917,38.404611,0.0,0.0
4,MA,21.104945,37.759183,21.104945,37.759183,0.0,0.0
5,MR,17.269683,32.469517,17.269683,32.469517,0.0,0.0
6,OM,29.185476,43.220394,29.185476,43.220394,0.0,0.0
7,PS,33.094902,47.532230,33.094902,47.532230,0.0,0.0
8,SA,37.314794,51.947894,37.314794,51.947894,0.0,0.0
9,SD,25.907977,40.972611,25.907977,40.972611,0.0,0.0


High-confidence fixes: 0
Actually changed translations: 0
Invalid correction fallbacks: 0
Macro spBLEU gain: +0.000000
Harmed countries: 0
Worst country delta: +0.000000

STOP — DO NOT RUN FULL LOCKED40 OR PRIVATE TEST
Saved: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1/system103_r2_locked40_fixed_subset_v1


In [6]:
# Diagnose why R2 produced zero eligible corrections.

diagnostic = planner.copy()

diagnostic["raw_planner"] = (
    diagnostic["raw_planner"]
    .fillna("")
    .astype(str)
)

diagnostic["has_think"] = (
    diagnostic["raw_planner"]
    .str.contains(
        "<think>",
        regex=False,
    )
)

diagnostic["closed_think"] = (
    diagnostic["raw_planner"]
    .str.contains(
        "</think>",
        regex=False,
    )
)

diagnostic["has_decision_schema"] = (
    diagnostic["raw_planner"]
    .str.contains(
        r"DECISION\s*:",
        case=False,
        regex=True,
    )
)

diagnostic["diagnosis"] = np.select(
    [
        diagnostic["has_think"]
        & ~diagnostic["closed_think"],

        ~diagnostic[
            "has_decision_schema"
        ],

        diagnostic["decision"]
        .eq("FIX")
        & diagnostic["confidence"]
        .isin(["LOW", "MEDIUM"]),

        diagnostic["decision"]
        .eq("KEEP"),
    ],
    [
        "incomplete_thinking",
        "missing_schema",
        "fix_below_high_confidence",
        "explicit_keep",
    ],
    default="other",
)

display(
    diagnostic.groupby(
        [
            "diagnosis",
            "decision",
            "confidence",
            "apply_fix",
        ],
        as_index=False,
    ).size()
)

print("\nRepresentative raw outputs:")

for diagnosis, part in diagnostic.groupby(
    "diagnosis",
    sort=False,
):
    print("\n" + "=" * 80)
    print(diagnosis)
    print("=" * 80)

    for _, row in part.head(2).iterrows():
        print(
            f"\n{row['country']} | "
            f"{row['source_id']} | "
            f"{row['decision']} | "
            f"{row['confidence']}"
        )

        print(
            row["raw_planner"][-1500:]
        )

,diagnosis,decision,confidence,apply_fix,size
0,missing_schema,KEEP,LOW,False,1056



Representative raw outputs:

missing_schema

EG | EG_test_B0-1-1-152_1 | KEEP | LOW
/you think اللحمة مستوية زيادة شكلها ناشفة اوي

EG | EG_test_B0-1-1-152_2 | KEEP | LOW
/ معلش، حضرتك عايزني اطلب من الشيف يعملك واحدة تانية؟


In [7]:
V2_DIR = OUT / "constrained_r2_v2"
V2_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_V2_PATH = V2_DIR / "audit_logits.csv"

KEEP_SUFFIX = " A"
FIX_SUFFIX = " B"
AUDIT_BATCH_SIZE = 4

print("Choice token IDs:", {
    "KEEP_A": tokenizer.encode(
        KEEP_SUFFIX,
        add_special_tokens=False,
    ),
    "FIX_B": tokenizer.encode(
        FIX_SUFFIX,
        add_special_tokens=False,
    ),
})


def build_forced_audit_prompt(row):
    return f"""### System:
You are a strict and conservative quality controller for
context-aware English-to-dialectal-Arabic translation.

### Instruction:
Judge whether the existing Arabic translation has a definite error.

Country/config: {clean_string(row["country"])}
Target dialect: {clean_string(row["dialect"])}
Domain: {clean_string(row["domain"])}
Participants/roles: {clean_string(row["participants"])}
Current speaker: {clean_string(row["speaker"])}
Speaker-to-addressee gender: {clean_string(row["gender_direction"])}

Previous English context:
{format_previous_context(row, include_speakers=True)}

Current English turn:
{clean_string(row["source_text"])}

Existing Arabic translation:
{clean_string(row["incumbent_prediction"])}

Choose exactly one:

A = KEEP. Meaning is preserved and the translation is acceptable.
B = FIX. A definite meaning, omission/addition, referent, gender,
entity, number, idiom, or target-dialect error requires correction.

Valid paraphrases, spelling alternatives and optional wording must be A.

### Decision:"""


@torch.inference_mode()
def score_keep_fix(prompts):
    options = [KEEP_SUFFIX, FIX_SUFFIX]

    sequences = []
    metadata = []

    max_option_tokens = max(
        len(
            tokenizer.encode(
                option,
                add_special_tokens=False,
            )
        )
        for option in options
    )

    for prompt_index, prompt in enumerate(prompts):
        prompt_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=(
                MAX_SEQ_LENGTH
                - max_option_tokens
            ),
        )

        for option_index, option in enumerate(options):
            option_ids = tokenizer.encode(
                option,
                add_special_tokens=False,
            )

            sequences.append(
                prompt_ids + option_ids
            )

            metadata.append({
                "prompt_index": prompt_index,
                "option_index": option_index,
                "prompt_length": len(prompt_ids),
                "option_ids": option_ids,
            })

    maximum_length = max(
        len(sequence)
        for sequence in sequences
    )

    input_ids = torch.full(
        (
            len(sequences),
            maximum_length,
        ),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros_like(
        input_ids
    )

    for index, sequence in enumerate(sequences):
        input_ids[
            index,
            -len(sequence):,
        ] = torch.tensor(sequence)

        attention_mask[
            index,
            -len(sequence):,
        ] = 1

    device = next(model.parameters()).device

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    ).logits.float()

    log_probs = torch.log_softmax(
        logits,
        dim=-1,
    )

    scores = np.full(
        (len(prompts), 2),
        np.nan,
        dtype=np.float64,
    )

    for row_index, info in enumerate(metadata):
        padding = (
            maximum_length
            - len(sequences[row_index])
        )

        option_start = (
            padding
            + info["prompt_length"]
        )

        token_scores = []

        for offset, token_id in enumerate(
            info["option_ids"]
        ):
            token_position = (
                option_start + offset
            )

            token_scores.append(
                log_probs[
                    row_index,
                    token_position - 1,
                    token_id,
                ].item()
            )

        scores[
            info["prompt_index"],
            info["option_index"],
        ] = float(np.mean(token_scores))

    probabilities = torch.softmax(
        torch.tensor(
            scores,
            dtype=torch.float64,
        ),
        dim=1,
    ).numpy()

    return [
        {
            "keep_logp": float(
                scores[index, 0]
            ),
            "fix_logp": float(
                scores[index, 1]
            ),
            "fix_margin": float(
                scores[index, 1]
                - scores[index, 0]
            ),
            "p_fix": float(
                probabilities[index, 1]
            ),
        }
        for index in range(len(prompts))
    ]


if AUDIT_V2_PATH.is_file():
    audit_v2 = pd.read_csv(
        AUDIT_V2_PATH,
        dtype={"source_id": str},
    )

    audit_v2 = (
        audit_v2[
            audit_v2["source_id"]
            .isin(set(subset["source_id"]))
        ]
        .drop_duplicates(
            "source_id",
            keep="last",
        )
    )

else:
    audit_v2 = pd.DataFrame()

completed_ids = (
    set(
        audit_v2["source_id"].astype(str)
    )
    if len(audit_v2)
    else set()
)

pending_records = []

for country in COUNTRIES:
    adapter_key = ROUTE[country][0]

    # The routed adapter is active while scoring KEEP/FIX.
    activate_adapter(adapter_key)

    country_rows = subset[
        subset["country"].eq(country)
        & ~subset["source_id"]
        .isin(completed_ids)
    ]

    for start in tqdm(
        range(
            0,
            len(country_rows),
            AUDIT_BATCH_SIZE,
        ),
        desc=f"Constrained audit {country}",
    ):
        batch = country_rows.iloc[
            start:start + AUDIT_BATCH_SIZE
        ]

        prompts = [
            build_forced_audit_prompt(row)
            for _, row in batch.iterrows()
        ]

        scores = score_keep_fix(prompts)

        for (_, row), score in zip(
            batch.iterrows(),
            scores,
        ):
            pending_records.append({
                "source_id": str(
                    row["source_id"]
                ),
                "country": country,
                "adapter_key": adapter_key,
                **score,
            })

        if len(pending_records) >= 24:
            audit_v2 = pd.concat(
                [
                    audit_v2,
                    pd.DataFrame(
                        pending_records
                    ),
                ],
                ignore_index=True,
            ).drop_duplicates(
                "source_id",
                keep="last",
            )

            atomic_csv(
                audit_v2,
                AUDIT_V2_PATH,
            )

            pending_records = []

if pending_records:
    audit_v2 = pd.concat(
        [
            audit_v2,
            pd.DataFrame(
                pending_records
            ),
        ],
        ignore_index=True,
    ).drop_duplicates(
        "source_id",
        keep="last",
    )

    atomic_csv(
        audit_v2,
        AUDIT_V2_PATH,
    )

if (
    set(
        audit_v2[
            "source_id"
        ].astype(str)
    )
    != set(
        subset[
            "source_id"
        ].astype(str)
    )
):
    raise RuntimeError(
        "Constrained audit incomplete. "
        "Rerun this cell to resume."
    )

# Rank independently inside each country.
audit_v2["country_rank"] = (
    audit_v2
    .groupby("country")["fix_margin"]
    .rank(
        method="first",
        ascending=False,
    )
)

audit_v2["country_rows"] = (
    audit_v2
    .groupby("country")["source_id"]
    .transform("size")
)

audit_v2["rank_fraction"] = (
    audit_v2["country_rank"]
    / audit_v2["country_rows"]
)

atomic_csv(
    audit_v2,
    AUDIT_V2_PATH,
)

display(
    audit_v2.groupby(
        "country",
        as_index=False,
    ).agg(
        rows=("source_id", "size"),
        mean_fix_margin=(
            "fix_margin",
            "mean",
        ),
        maximum_fix_margin=(
            "fix_margin",
            "max",
        ),
        mean_p_fix=("p_fix", "mean"),
        p_fix_above_half=(
            "p_fix",
            lambda values: int(
                (values > 0.5).sum()
            ),
        ),
    ).round(6)
)

print("Overall FIX-margin distribution:")

display(
    audit_v2[
        "fix_margin"
    ].describe(
        percentiles=[
            0.50,
            0.80,
            0.90,
            0.95,
            0.975,
        ]
    ).to_frame()
)

Choice token IDs: {'KEEP_A': [362], 'FIX_B': [425]}


Constrained audit EG:   0%|          | 0/21 [00:00<?, ?it/s]

Constrained audit JO:   0%|          | 0/20 [00:00<?, ?it/s]

Constrained audit LB:   0%|          | 0/20 [00:00<?, ?it/s]

Constrained audit LY:   0%|          | 0/20 [00:00<?, ?it/s]

...

Constrained audit TN:   0%|          | 0/21 [00:00<?, ?it/s]

Constrained audit YE:   0%|          | 0/21 [00:00<?, ?it/s]

,country,rows,mean_fix_margin,maximum_fix_margin,mean_p_fix,p_fix_above_half
0,EG,81,-1.295525,0.0000,0.233331,0
1,JO,80,-0.914062,0.2500,0.298210,3
2,LB,80,-0.722656,0.7500,0.343168,8
3,LY,80,-1.796094,0.1250,0.198028,1
4,MA,81,-0.148148,1.1250,0.465806,28
5,MR,83,-0.381777,0.5000,0.412777,11
6,OM,81,-0.703704,0.6250,0.347947,10
7,PS,81,-1.009259,0.2500,0.286876,1
8,SA,82,-0.653201,0.8750,0.353311,5
9,SD,83,-1.082831,0.7500,0.282845,6


Overall FIX-margin distribution:


,fix_margin
count,1056.000000
mean,-0.904179
std,0.865553
min,-5.625000
50%,-0.750000
80%,-0.250000
90%,0.000000
95%,0.250000
97.5%,0.375000
max,1.125000


In [8]:
CORRECTIONS_V2_PATH = (
    V2_DIR / "corrections_top20.csv"
)

MAX_GENERATED_FRACTION = 0.20

candidate_audit = audit_v2[
    audit_v2["rank_fraction"]
    .le(MAX_GENERATED_FRACTION)
].copy()

candidate_ids = set(
    candidate_audit[
        "source_id"
    ].astype(str)
)

candidate_rows = subset[
    subset["source_id"]
    .isin(candidate_ids)
].copy()


def build_v2_correction_prompt(row):
    direct_prompt = build_prompt(
        row,
        variant=ROUTE[
            str(row["country"])
        ][2],
        retrieval_mode="new",
    )

    direct_prompt = direct_prompt.rsplit(
        "### Arabic translation:",
        1,
    )[0].rstrip()

    return f"""{direct_prompt}

Existing Arabic draft:
{clean_string(row["incumbent_prediction"])}

The quality controller ranked this draft as potentially
containing a clear error.

Recheck the English turn, dialogue context, metadata and
existing draft yourself.

Rules:
- If there is a definite error, make only the smallest necessary correction.
- Do not paraphrase unaffected wording.
- If the draft is acceptable, reproduce it exactly.
- Return only the final Arabic translation.

### Arabic translation:
"""


def parse_v2_correction(row, output):
    return {
        "prediction": str(output).strip()
    }


corrections_v2 = run_routed_resumable(
    target=candidate_rows,
    output_path=CORRECTIONS_V2_PATH,
    prompt_function=build_v2_correction_prompt,
    generation_function=generate_batch,
    parser_function=parse_v2_correction,
    batch_size=GEN_BATCH_SIZE,
)

print(
    "Top-20% correction candidates:",
    len(candidate_rows),
)

print(
    "Corrections completed:",
    len(corrections_v2),
)

corrections_top20 EG:   0%|          | 0/8 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


corrections_top20 JO:   0%|          | 0/8 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


corrections_top20 LB:   0%|          | 0/8 [00:00<?, ?it/s]

...

corrections_top20 TN:   0%|          | 0/8 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


corrections_top20 YE:   0%|          | 0/8 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Top-20% correction candidates: 208
Corrections completed: 208


In [9]:
GATES = [
    0.025,
    0.05,
    0.10,
    0.15,
    0.20,
]

correction_map_v2 = (
    corrections_v2
    .set_index("source_id")[
        "prediction"
    ]
)

gate_records = []
gate_predictions = {}

for gate in GATES:
    selected_ids = set(
        audit_v2.loc[
            audit_v2[
                "rank_fraction"
            ].le(gate),
            "source_id",
        ].astype(str)
    )

    frame = subset[
        [
            "source_id",
            "country",
            "reference_arabic",
            "incumbent_prediction",
        ]
    ].copy()

    frame["prediction"] = (
        frame["incumbent_prediction"]
    )

    frame["invalid_fallback"] = False

    for index, row in frame.iterrows():
        source_id = str(
            row["source_id"]
        )

        if source_id not in selected_ids:
            continue

        candidate = str(
            correction_map_v2.get(
                source_id,
                "",
            )
        ).strip()

        if (
            candidate
            and ARABIC_PATTERN.search(
                candidate
            )
        ):
            frame.at[
                index,
                "prediction",
            ] = candidate

        else:
            frame.at[
                index,
                "invalid_fallback",
            ] = True

    frame["changed"] = (
        frame["prediction"]
        .astype(str)
        .str.strip()
        != frame[
            "incumbent_prediction"
        ]
        .astype(str)
        .str.strip()
    )

    scored = score_system(
        frame,
        "prediction",
        f"R2_v2_top_{gate:.3f}",
    )

    country_delta = (
        scored[
            [
                "country",
                "spBLEU",
                "chrF++",
            ]
        ]
        .merge(
            incumbent_metrics[
                [
                    "country",
                    "spBLEU",
                    "chrF++",
                ]
            ],
            on="country",
            suffixes=(
                "_R2",
                "_incumbent",
            ),
            validate="one_to_one",
        )
    )

    country_delta["spBLEU_delta"] = (
        country_delta["spBLEU_R2"]
        - country_delta[
            "spBLEU_incumbent"
        ]
    )

    country_delta["chrF++_delta"] = (
        country_delta["chrF++_R2"]
        - country_delta[
            "chrF++_incumbent"
        ]
    )

    gate_records.append({
        "gate_fraction": gate,
        "selected": len(selected_ids),
        "changed": int(
            frame["changed"].sum()
        ),
        "invalid_fallbacks": int(
            frame[
                "invalid_fallback"
            ].sum()
        ),
        "macro_spBLEU": float(
            scored["spBLEU"].mean()
        ),
        "macro_chrF++": float(
            scored["chrF++"].mean()
        ),
        "spBLEU_gain": float(
            scored["spBLEU"].mean()
            - incumbent_bleu
        ),
        "chrF++_gain": float(
            scored["chrF++"].mean()
            - incumbent_chrf
        ),
        "harmed_countries": int(
            country_delta[
                "spBLEU_delta"
            ].lt(-1e-9).sum()
        ),
        "worst_country_delta": float(
            country_delta[
                "spBLEU_delta"
            ].min()
        ),
    })

    gate_predictions[
        round(gate, 3)
    ] = frame

gate_results = (
    pd.DataFrame(gate_records)
    .sort_values(
        [
            "spBLEU_gain",
            "chrF++_gain",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

best = gate_results.iloc[0]

best_gate = round(
    float(
        best["gate_fraction"]
    ),
    3,
)

best_frame = gate_predictions[
    best_gate
]

passed = (
    float(best["spBLEU_gain"]) >= 0.20
    and int(
        best["harmed_countries"]
    ) <= 4
    and float(
        best["worst_country_delta"]
    ) >= -0.60
)

display(
    gate_results.round(6)
)

print("Best gate:", best_gate)

print(
    f"Gain {best['spBLEU_gain']:+.6f} | "
    f"chrF++ {best['chrF++_gain']:+.6f} | "
    f"harmed "
    f"{int(best['harmed_countries'])} | "
    f"worst "
    f"{best['worst_country_delta']:+.6f}"
)

print()

print(
    "PASS — APPLY THIS GATE TO FULL LOCKED40"
    if passed
    else
    "STOP — ABANDON REASONING; "
    "DO NOT RUN R1"
)

atomic_csv(
    gate_results,
    V2_DIR / "gate_results.csv",
)

atomic_csv(
    best_frame,
    V2_DIR / "best_gate_predictions.csv",
)

(
    V2_DIR / "best_gate.json"
).write_text(
    json.dumps(
        {
            "best_gate_fraction": (
                best_gate
            ),
            "spBLEU_gain": float(
                best["spBLEU_gain"]
            ),
            "chrF++_gain": float(
                best["chrF++_gain"]
            ),
            "harmed_countries": int(
                best[
                    "harmed_countries"
                ]
            ),
            "worst_country_delta": float(
                best[
                    "worst_country_delta"
                ]
            ),
            "passed": bool(passed),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", V2_DIR)

,gate_fraction,selected,changed,invalid_fallbacks,macro_spBLEU,macro_chrF++,spBLEU_gain,chrF++_gain,harmed_countries,worst_country_delta
0,0.050,52,7,0,29.705364,44.325063,0.003316,0.006085,2,-0.159568
1,0.025,26,0,0,29.702048,44.318978,0.000000,0.000000,0,0.000000
2,0.150,156,20,0,29.698995,44.308166,-0.003054,-0.010812,4,-0.159568
3,0.100,104,15,0,29.696389,44.309525,-0.005659,-0.009453,4,-0.159568
4,0.200,208,25,0,29.678597,44.295457,-0.023452,-0.023521,4,-0.314558


Best gate: 0.05
Gain +0.003316 | chrF++ +0.006085 | harmed 2 | worst -0.159568

STOP — ABANDON REASONING; DO NOT RUN R1
Saved: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1/system103_r2_locked40_fixed_subset_v1/constrained_r2_v2


---

### **New Experiment**

In [1]:
from pathlib import Path
import ast, json, os, re, unicodedata
import numpy as np
import pandas as pd
import sacrebleu
from IPython.display import display

ROOT = Path("/home/mabdallah/alexandriax_mt_14d")
RUN = (
    "nilechat3b_dev12250_hftest60_from16600_complete2shot_"
    "r16_alpha32_2epochs_nonquant_server5090_v1"
)
DATA = ROOT / "data/nilechat3b_dev_continuation" / RUN
STAGE = ROOT / "variant_selection_competition_v1"
OUT = STAGE / "system104_locked40_exact_tm_v1"
OUT.mkdir(parents=True, exist_ok=True)

LOCKED_PATH = DATA / "selection_prompt_rows.pkl"
MEMORY_PATH = DATA / "fine_tune_prompt_rows.pkl"  # train60 + DEV cache

ROUTES = {
    "EG":"c5200_v05", "JO":"c5200_v01", "LB":"c5200_v01",
    "LY":"c6400_v05", "MA":"c5200_v03", "MR":"c2000_v01",
    "OM":"c4900_v01", "PS":"c5200_v01", "SA":"c1600_v03",
    "SD":"c5200_v01", "SY":"c5200_v05", "TN":"c7600_v01",
    "YE":"c4900_v01",
}
OVERLAYS = {
    "LY":"extspec_ly_step000400_v05_exact_submission_route_v5",
    "SD":"extspec_sd_step000700_v01_exact_submission_route_v5",
}

def canon(df, locked=False):
    df = df.copy()
    if "country" not in df:
        df["country"] = df["config"]
    if "target_arabic" not in df:
        df["target_arabic"] = df["reference_arabic"]

    for column, default in {
        "previous_english_turns": [],
        "domain": "",
        "speaker": "",
        "gender_direction": "",
    }.items():
        if column not in df:
            df[column] = [default for _ in range(len(df))]

    required = {
        "country", "source_id", "conversation_id",
        "source_text", "target_arabic",
    }
    if required - set(df):
        raise KeyError(f"Missing columns: {required - set(df)}")

    df["country"] = df["country"].astype(str).str.strip().str.upper()

    for column in [
        "source_id", "conversation_id", "source_text", "target_arabic",
        "domain", "speaker", "gender_direction",
    ]:
        df[column] = df[column].fillna("").astype(str)

    if locked:
        df = df.reset_index(drop=True)
        df["_row_order"] = np.arange(len(df))
        df["reference_arabic"] = df["target_arabic"]

    return df

def prediction_file(experiment):
    for name in ["turn_predictions.csv", "scored_turn_predictions.csv"]:
        path = STAGE / experiment / name
        if path.is_file():
            return path
    raise FileNotFoundError(STAGE / experiment)

def load_predictions(experiment, required_ids):
    frame = pd.read_csv(
        prediction_file(experiment),
        dtype={"source_id": str},
        keep_default_na=False,
    )
    prediction_column = next(
        (
            c for c in [
                "prediction", "generated_translation", "predicted_arabic"
            ] if c in frame
        ),
        None,
    )
    if prediction_column is None:
        raise KeyError(f"{experiment}: prediction column missing")

    frame = (
        frame.loc[
            frame["source_id"].isin(required_ids),
            ["source_id", prediction_column],
        ]
        .rename(columns={prediction_column: "prediction"})
        .drop_duplicates("source_id", keep="last")
    )

    if (
        set(frame["source_id"]) != set(required_ids)
        or frame["prediction"].astype(str).str.strip().eq("").any()
    ):
        raise RuntimeError(f"{experiment}: incomplete or empty predictions")

    return frame

def score(frame, prediction_column="prediction"):
    records = []

    for country, part in (
        frame.sort_values("_row_order").groupby("country", sort=True)
    ):
        hypotheses = part[prediction_column].astype(str).tolist()
        references = part["reference_arabic"].astype(str).tolist()

        records.append({
            "country": country,
            "rows": len(part),
            "spBLEU": sacrebleu.corpus_bleu(
                hypotheses, [references], tokenize="flores200"
            ).score,
            "chrF++": sacrebleu.corpus_chrf(
                hypotheses, [references], word_order=2
            ).score,
        })

    per_country = pd.DataFrame(records)

    return per_country, {
        "macro_spBLEU": float(per_country["spBLEU"].mean()),
        "macro_chrF++": float(per_country["chrF++"].mean()),
    }

locked = canon(pd.read_pickle(LOCKED_PATH), locked=True)
memory = canon(pd.read_pickle(MEMORY_PATH))

assert len(locked) == 5772
assert locked["source_id"].is_unique
assert set(memory["source_id"]).isdisjoint(set(locked["source_id"]))
assert set(zip(memory["country"], memory["conversation_id"])).isdisjoint(
    set(zip(locked["country"], locked["conversation_id"]))
)

parts = []

for country, experiment in ROUTES.items():
    ids = set(locked.loc[locked["country"].eq(country), "source_id"])
    parts.append(load_predictions(experiment, ids))

incumbent_predictions = pd.concat(parts, ignore_index=True)

for country, experiment in OVERLAYS.items():
    ids = set(locked.loc[locked["country"].eq(country), "source_id"])
    overlay = (
        load_predictions(experiment, ids)
        .set_index("source_id")["prediction"]
    )
    mask = incumbent_predictions["source_id"].isin(ids)
    incumbent_predictions.loc[mask, "prediction"] = (
        incumbent_predictions.loc[mask, "source_id"].map(overlay)
    )

incumbent = (
    locked
    .merge(incumbent_predictions, on="source_id", validate="one_to_one")
    .sort_values("_row_order")
    .reset_index(drop=True)
)

incumbent_per_country, incumbent_score = score(incumbent)

if abs(incumbent_score["macro_spBLEU"] - 30.317100) > 0.002:
    raise RuntimeError(f"Wrong incumbent reconstructed: {incumbent_score}")

print(f"Locked40 rows: {len(locked)}")
print(f"Memory rows:   {len(memory)}")
print(
    f"INCUMBENT: {incumbent_score['macro_spBLEU']:.6f} spBLEU | "
    f"{incumbent_score['macro_chrF++']:.6f} chrF++"
)

Locked40 rows: 5772
Memory rows:   20920
INCUMBENT: 30.317100 spBLEU | 44.898931 chrF++


In [2]:
TRANS = str.maketrans({
    "’":"'", "‘":"'", "“":'"', "”":'"',
    "–":"-", "—":"-", "−":"-", "…":"...",
})
DIACRITICS = re.compile(
    r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]"
)
GENERIC = {
    "yes", "no", "ok", "okay", "sure", "right", "really",
    "thanks", "thank you", "hello", "hi", "bye", "goodbye", "please",
}

def safe_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)

def normalize_english(value):
    text = (
        unicodedata.normalize("NFKC", safe_text(value))
        .translate(TRANS)
        .lower()
    )
    text = re.sub(r"\s*([,.;:!?()\[\]{}])\s*", r"\1 ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_metadata(value):
    return re.sub(
        r"\s+", " ",
        unicodedata.normalize("NFKC", safe_text(value)).lower(),
    ).strip()

def normalize_arabic(value):
    text = unicodedata.normalize("NFKC", safe_text(value))
    text = DIACRITICS.sub("", text).replace("ـ", "")
    return re.sub(r"\s+", " ", text).strip()

def parse_turns(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, dict):
        return [value]
    if isinstance(value, (list, tuple)):
        return list(value)

    text = safe_text(value).strip()
    if not text or text.lower() in {"nan", "none", "[]"}:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return [parsed]
            if isinstance(parsed, (list, tuple)):
                return list(parsed)
        except Exception:
            pass

    return [text]

def context_key(value):
    parts = []

    for turn in parse_turns(value)[-3:]:
        if isinstance(turn, dict):
            speaker = normalize_metadata(turn.get("speaker", ""))
            text = normalize_english(
                turn.get(
                    "text",
                    turn.get("source_text", turn.get("english", "")),
                )
            )
        else:
            speaker, text = "", normalize_english(turn)

        if text:
            parts.append(f"{speaker}\u241e{text}")

    return "\u241f".join(parts)

def informative_source(key, minimum_words=4, minimum_chars=15):
    words = re.findall(r"[a-z0-9]+(?:'[a-z]+)?", key)
    plain = re.sub(r"[^a-z0-9]", "", key)

    return (
        key.strip(" .,!?:;'\"") not in GENERIC
        and len(words) >= minimum_words
        and len(plain) >= minimum_chars
    )

for frame in [memory, locked]:
    frame["_src"] = frame["source_text"].map(normalize_english)
    frame["_ctx"] = frame["previous_english_turns"].map(context_key)
    frame["_dom"] = frame["domain"].map(normalize_metadata)
    frame["_gen"] = frame["gender_direction"].map(normalize_metadata)
    frame["_spk"] = frame["speaker"].map(normalize_metadata)
    frame["_ar"] = frame["target_arabic"].map(normalize_arabic)
    frame["_words"] = frame["_src"].map(
        lambda x: len(re.findall(r"[a-z0-9]+(?:'[a-z]+)?", x))
    )
    frame["_meta"] = (
        frame[["_dom", "_gen", "_spk"]].ne("").sum(axis=1)
    )

memory = memory.loc[
    memory["_src"].ne("") & memory["_ar"].ne("")
].copy()

def safe_groups(frame, keys, acceptance_rule):
    records, rejected = [], 0

    for key, group in frame.groupby(keys, sort=False, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        counts = group["_ar"].value_counts()
        support = int(counts.sum())

        stats = {
            "support": support,
            "variants": len(counts),
            "words": int(group["_words"].iloc[0]),
            "meta": int(group["_meta"].iloc[0]),
            "agreement": float(counts.iloc[0] / support),
        }

        if not acceptance_rule(stats):
            rejected += 1
            continue

        top_normalized_target = counts.index[0]
        target = (
            group.loc[
                group["_ar"].eq(top_normalized_target),
                "target_arabic",
            ]
            .astype(str)
            .str.strip()
            .value_counts()
            .index[0]
        )

        records.append({
            **dict(zip(keys, key)),
            "tm_candidate": target,
            "tm_support": support,
            "tm_agreement": stats["agreement"],
        })

    return pd.DataFrame(records), rejected

base = memory.loc[memory["_src"].map(informative_source)]

# T1: exact source + exact non-empty English context.
t1, rejected_t1 = safe_groups(
    base.loc[base["_ctx"].ne("")],
    ["country", "_src", "_ctx"],
    lambda s: s["variants"] == 1,
)

# T2: exact source + strong metadata compatibility.
t2, rejected_t2 = safe_groups(
    base.loc[base["_meta"].ge(2)],
    ["country", "_src", "_dom", "_gen", "_spk"],
    lambda s: (
        s["variants"] == 1
        and (
            s["support"] >= 2
            or (s["meta"] == 3 and s["words"] >= 7)
        )
    ),
)

# T3: exact source repeated at least 3 times with one unanimous target.
t3, rejected_t3 = safe_groups(
    base,
    ["country", "_src"],
    lambda s: s["variants"] == 1 and s["support"] >= 3,
)

TIER_SPECS = {
    "T1": (t1, ["country", "_src", "_ctx"]),
    "T2": (t2, ["country", "_src", "_dom", "_gen", "_spk"]),
    "T3": (t3, ["country", "_src"]),
}

MATCHES = {}

for tier, (table, keys) in TIER_SPECS.items():
    if table.empty:
        MATCHES[tier] = pd.DataFrame(columns=[
            "source_id", "tm_candidate", "tm_support", "tm_agreement"
        ])
    else:
        MATCHES[tier] = (
            locked[["source_id", *keys]]
            .merge(table, on=keys, how="inner", validate="many_to_one")
            [["source_id", "tm_candidate", "tm_support", "tm_agreement"]]
        )

display(pd.DataFrame({
    "tier": [
        "T1 exact context",
        "T2 exact metadata",
        "T3 unanimous source",
    ],
    "locked40_matches": [
        len(MATCHES["T1"]),
        len(MATCHES["T2"]),
        len(MATCHES["T3"]),
    ],
    "rejected_memory_groups": [
        rejected_t1,
        rejected_t2,
        rejected_t3,
    ],
}))

,tier,locked40_matches,rejected_memory_groups
0,T1 exact context,0,0
1,T2 exact metadata,0,559
2,T3 unanimous source,0,20803


In [3]:
RULES = {
    "T1_exact_context": ["T1"],
    "T1_T2_exact_metadata": ["T1", "T2"],
    "T1_T2_T3_unanimous": ["T1", "T2", "T3"],
}

def save_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

summaries = {}
products = {}

for rule_name, tier_order in RULES.items():
    candidate = incumbent.copy()
    candidate["incumbent_prediction"] = candidate["prediction"]
    candidate["tm_candidate"] = None
    candidate["tm_tier"] = ""
    candidate["tm_support"] = np.nan

    for tier in tier_order:
        matches = (
            MATCHES[tier]
            .drop_duplicates("source_id")
            .set_index("source_id")
        )

        if matches.empty:
            continue

        mapped = candidate["source_id"].map(matches["tm_candidate"])
        use = candidate["tm_candidate"].isna() & mapped.notna()

        candidate.loc[use, "tm_candidate"] = mapped[use]
        candidate.loc[use, "tm_tier"] = tier
        candidate.loc[use, "tm_support"] = (
            candidate.loc[use, "source_id"].map(matches["tm_support"])
        )

    selected = candidate["tm_candidate"].notna()
    candidate.loc[selected, "prediction"] = (
        candidate.loc[selected, "tm_candidate"]
    )

    candidate["tm_changed"] = (
        selected
        & candidate["prediction"].astype(str).str.strip().ne(
            candidate["incumbent_prediction"].astype(str).str.strip()
        )
    )

    per_country, overall = score(candidate)

    deltas = per_country.merge(
        incumbent_per_country,
        on=["country", "rows"],
        suffixes=("_candidate", "_incumbent"),
        validate="one_to_one",
    )
    deltas["spBLEU_delta"] = (
        deltas["spBLEU_candidate"] - deltas["spBLEU_incumbent"]
    )
    deltas["chrF++_delta"] = (
        deltas["chrF++_candidate"] - deltas["chrF++_incumbent"]
    )

    output_dir = OUT / rule_name

    save_csv(
        candidate[[
            "source_id", "country", "prediction",
            "incumbent_prediction", "tm_candidate",
            "tm_tier", "tm_support", "tm_changed", "_row_order",
        ]],
        output_dir / "turn_predictions.csv",
    )
    save_csv(deltas, output_dir / "per_country_deltas.csv")

    summaries[rule_name] = {
        "rule": rule_name,
        "selected": int(selected.sum()),
        "changed": int(candidate["tm_changed"].sum()),
        "tiers": json.dumps(
            candidate.loc[selected, "tm_tier"].value_counts().to_dict()
        ),
        **overall,
        "spBLEU_gain": (
            overall["macro_spBLEU"] - incumbent_score["macro_spBLEU"]
        ),
        "chrF++_gain": (
            overall["macro_chrF++"] - incumbent_score["macro_chrF++"]
        ),
        "harmed_countries": int(
            deltas["spBLEU_delta"].lt(-1e-12).sum()
        ),
        "worst_country_delta": float(
            deltas["spBLEU_delta"].min()
        ),
    }

    products[rule_name] = (candidate, deltas)

leaderboard = (
    pd.DataFrame(summaries.values())
    .sort_values(
        ["macro_spBLEU", "macro_chrF++"],
        ascending=False,
    )
    .reset_index(drop=True)
)

save_csv(leaderboard, OUT / "rule_leaderboard.csv")
display(leaderboard)

best = leaderboard.iloc[0]
best_candidate, best_deltas = products[best["rule"]]

display(
    best_deltas[[
        "country",
        "spBLEU_candidate",
        "spBLEU_incumbent",
        "spBLEU_delta",
        "chrF++_delta",
    ]]
    .sort_values("spBLEU_delta", ascending=False)
    .reset_index(drop=True)
)

passed = (
    best["spBLEU_gain"] >= 0.10
    and best["chrF++_gain"] >= 0.0
    and best["harmed_countries"] <= 2
    and best["worst_country_delta"] >= -0.15
)

manifest = {
    "pass": bool(passed),
    "incumbent": incumbent_score,
    **best.to_dict(),
}

(OUT / "promotion_gate.json").write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 86)
print("PROMOTION GATE:", "PASS" if passed else "FAIL")
print(f"Best rule: {best['rule']}")
print(f"spBLEU gain: {best['spBLEU_gain']:+.6f} [required >= +0.10]")
print(f"chrF++ gain: {best['chrF++_gain']:+.6f} [required >= 0.00]")
print(f"Harmed countries: {int(best['harmed_countries'])} [required <= 2]")
print(
    f"Worst country delta: {best['worst_country_delta']:+.6f} "
    "[required >= -0.15]"
)
print("Saved:", OUT)

print(
    "NEXT: build the private-test overlay"
    if passed
    else "STOP: retain incumbent; do not alter private-test predictions"
)

,rule,selected,changed,tiers,macro_spBLEU,macro_chrF++,spBLEU_gain,chrF++_gain,harmed_countries,worst_country_delta
0,T1_exact_context,0,0,{},30.3171,44.898931,0.0,0.0,0,0.0
1,T1_T2_exact_metadata,0,0,{},30.3171,44.898931,0.0,0.0,0,0.0
2,T1_T2_T3_unanimous,0,0,{},30.3171,44.898931,0.0,0.0,0,0.0


,country,spBLEU_candidate,spBLEU_incumbent,spBLEU_delta,chrF++_delta
0,EG,31.875345,31.875345,0.0,0.0
1,JO,35.499177,35.499177,0.0,0.0
2,LB,32.288100,32.288100,0.0,0.0
3,LY,23.855589,23.855589,0.0,0.0
4,MA,23.306449,23.306449,0.0,0.0
5,MR,17.940801,17.940801,0.0,0.0
6,OM,32.574507,32.574507,0.0,0.0
7,PS,34.256767,34.256767,0.0,0.0
8,SA,35.245040,35.245040,0.0,0.0
9,SD,27.433504,27.433504,0.0,0.0



PROMOTION GATE: FAIL
Best rule: T1_exact_context
spBLEU gain: +0.000000 [required >= +0.10]
chrF++ gain: +0.000000 [required >= 0.00]
Harmed countries: 0 [required <= 2]
Worst country delta: +0.000000 [required >= -0.15]
Saved: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1/system104_locked40_exact_tm_v1
STOP: retain incumbent; do not alter private-test predictions
